# 01 — Collect a Training Dataset

This is the first of three notebooks that build a working MOUSE agent end-to-end:

| Notebook | What it does |
|---|---|
| **01 — Collect** (this one) | Run Procedural Frozen Lake with a continuous curriculum policy and save the transitions (including expert Q-star labels) to the Hub |
| **02 — Train** | Load those transitions and train a small model with a supervised policy (SP) objective driven by the Q-star labels |
| **03 — Inference** | Run the trained model live and watch it solve the maze |

MOUSE organises data into named `Datastore`s — one per environment slot — which map directly to Hugging Face dataset configs. That makes loading a specific environment's data by name straightforward in the training notebook.

### Why a curriculum?

A pure-random dataset rarely reaches the goal, so offline DQN or supervised policy training sees almost no positive reward signal. Mixing in oracle trajectories (actions selected by the exact optimal Q-table) ensures the dataset contains successful task completions that the model can learn from.

Rather than discrete phases, we use a **continuous linear ramp**: `oracle_prob` rises smoothly from 0 → 1 over all steps. At step `t` out of `T` total steps:

```
oracle_prob(t) = t / (T − 1)
```

The first step is fully random (broad exploration); the last step is fully oracle (pure expert). Every step in between blends both, so the transition is gradual with no abrupt jumps.

In [1]:
import numpy as np
import torch

from mouse_envs import EnvConfig, make_env
from mouse_core.data import Datastore, push_stores_to_hub

DATASET_ID  = "mouse-example-dataset"  # created under your authenticated HF user
TOTAL_STEPS = 1500                     # steps per slot; oracle_prob ramps 0 → 1 linearly

## Datastores

A `Datastore` is a sequential container for step data backed by a Hugging Face `Dataset` (Arrow). It stores rows — plain Python dicts — in the exact order they arrive. There is no required schema: each row contains whatever your collection script produced.

- `store.append(row)` adds a single step during collection.
- `store.append(other_store)` / `store.append([stores])` merges from another store.
- `Datastore.from_dataset(hf_dataset)` wraps an existing HF dataset.

The `name` given to a store becomes its config identifier on the Hub, which is how `load_stores_from_hub` retrieves it by name in the training notebook.

## Environment

`Procedural-FrozenLake-v1` is a parameterised variant of Gymnasium's FrozenLake that generates valid maze layouts and — crucially — can run value iteration on each reset to produce an optimal Q-table, exposed as `q_star` in the step output.

Each of the 4 slots generates its own unique 4×4 map from its seed (`seed+0`, `seed+1`, `seed+2`, `seed+3`), giving diverse hole layouts and distinct optimal policies across subsets. All slots share the same `Discrete(16)` observation space (4×4 = 16 states), so the training notebook can mix them without any extra configuration. Setting `q_star_source={"provider": "metadata_q_star"}` tells `make_env` to run the solver automatically and write the results into every step record.

In [2]:
config = EnvConfig(
    "Procedural-FrozenLake-v1",
    name="proc_frozenlake",
    seed=0,
    num_envs=4,
    episodes_per_task=10,
    kwargs={
        "is_slippery": False,
        "min_width": 4, "max_width": 4,
        "min_height": 4, "max_height": 4,
        "step_penalty": -0.05,
    },
    q_star_source={"provider": "metadata_q_star"},
)
env = make_env(config)
print("Slots:", env.names)

Slots: ('proc_frozenlake_0', 'proc_frozenlake_1', 'proc_frozenlake_2', 'proc_frozenlake_3')


## Collect transitions

Each row stores the **input** (action we sent) merged with the **output** (observation, reward, done code, time, and the expert `q_star` vector). The `q_star` field is a length-4 float array with the optimal Q-value for each action at the current state — used only as a supervision label during training, never fed back to the model at inference.

At each step `t`, `oracle_prob = t / (TOTAL_STEPS − 1)` is computed and each slot independently samples:
- with probability `(1 − oracle_prob)` → random action
- with probability `oracle_prob` → `argmax(q_star)` from the previous step's output

`collect` handles the initial reset internally — it steps the env once to obtain the first observations (the env ignores the action on that frame), stores the reset frame, then starts the ramp.

In [3]:
def pick_inputs(outputs, env, oracle_prob: float) -> list[dict]:
    """Choose actions for the next step.

    oracle_prob=0  → fully random
    oracle_prob=1  → always argmax(q_star)
    Each slot draws independently, so diversity is preserved across slots.
    """
    random_inputs = env.sample_random_inputs()
    inputs = []
    for i, out in enumerate(outputs):
        if np.random.random() >= oracle_prob:
            inputs.append(random_inputs[i])
        else:
            q = out.get("q_star")
            if q is not None:
                action = int(np.argmax(np.asarray(q)))
            else:
                action = int(random_inputs[i]["action"])
            inputs.append({"action": torch.tensor(action, dtype=torch.int64)})
    return inputs


def collect(stores, env, total_steps: int) -> None:
    """Collect total_steps steps per slot with a linear oracle ramp.

    Triggers the initial env reset internally, then ramps oracle_prob from
    0.0 (step 0, fully random) to 1.0 (step total_steps-1, fully oracle).
    """
    # First step triggers the reset; the env ignores the input and returns
    # the initial observation. Store it so the dataset starts from time=0.
    inputs = env.sample_random_inputs()
    outputs = env.step(inputs)
    for store, inp, out in zip(stores, inputs, outputs):
        store.append({**inp, **out})

    for t in range(total_steps):
        oracle_prob = t / (total_steps - 1)
        inputs = pick_inputs(outputs, env, oracle_prob)
        outputs = env.step(inputs)
        for store, inp, out in zip(stores, inputs, outputs):
            store.append({**inp, **out})

In [4]:
stores = [Datastore(name=name) for name in env.names]
collect(stores, env, TOTAL_STEPS)
env.close()

for store in stores:
    print(store)

Datastore(name='proc_frozenlake_0', steps=1501)
Datastore(name='proc_frozenlake_1', steps=1501)
Datastore(name='proc_frozenlake_2', steps=1501)
Datastore(name='proc_frozenlake_3', steps=1501)


## Push to the Hub

Each `Datastore` is uploaded as a separate named config inside the dataset repo. Anyone with the repo ID can call `load_stores_from_hub` to retrieve them by name.

In [5]:
url = push_stores_to_hub(stores, repo_id=DATASET_ID, split="train", private=False)
print(f"Pushed to {url}")

Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.


Processing Files (0 / 0): |          |  0.00B /  0.00B            

...

New Data Upload: |          |  0.00B /  0.00B            

Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Pushed to micahr234/mouse-example-dataset (proc_frozenlake_0/train: 1501, proc_frozenlake_1/train: 1501, proc_frozenlake_2/train: 1501, proc_frozenlake_3/train: 1501 steps)
Pushed to https://huggingface.co/datasets/micahr234/mouse-example-dataset
